# Parameter Sweep — DINOv2 (grid + hasil tersimpan)

Mencoba banyak kombinasi CONFIG sekaligus, lalu menyimpan:

- **`sweep_results/ac_predictions.csv`** — tabel prediksi AC (baris = foto, kolom = config)
- **`sweep_results/ac_heatmap.png`** — ringkasan visual foto × config
- **`sweep_results/ranking.csv` + `ranking.png`** — config diурут by weak-label Spearman (kompas)
- **`sweep_results/compare_XX.png`** — per foto: prediksi AC dari semua config berdampingan

**Caching pintar:** `temperature` tidak butuh re-embed; embedding di-cache per
(encoder, pooling, img_size, ROI, tta). Jadi 24 config default = hanya 8 pass embedding.

Jalankan **Run All**. Butuh `pip install torch timm scikit-learn scipy matplotlib pandas pillow pillow-heif`.

## 1. Grid parameter

Default = **smart reduced** (~24 config, jalan dalam menit). Mau lebih luas? tambah nilai ke list
mana pun di `GRID` (mis. `"img_size":[224,294]` atau `"roi_pad":[0.05,0.10,0.15]`) — tapi ingat
jumlah kombinasi mengalikan dan `encoder` besar (vitl14) berat.

In [ ]:
import glob, os, itertools, math
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import torch
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except ImportError:
    print("(opsional) pip install pillow-heif untuk file .heic")

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print("Device:", DEVICE)

TEST_DIR = "Front Teeth drg Laura"
_exts = ("*.png","*.jpg","*.jpeg","*.JPG","*.JPEG","*.PNG","*.heic","*.HEIC")
paths = sorted(set(sum([glob.glob(os.path.join(TEST_DIR, e)) for e in _exts], [])))
paths = [p for p in paths if not p.lower().endswith(".heic")] if False else paths  # heic dibuang otomatis kalau gagal load nanti
names = [os.path.splitext(os.path.basename(p))[0] for p in paths]
anchor_paths  = sorted(glob.glob("ac_references/ac_grade_*.png"))
anchor_grades = list(range(1, len(anchor_paths)+1))

# ===== GRID (ubah di sini) =====
GRID = {
    "encoder":     ["dinov2_vits14", "dinov2_vitb14"],   # + "dinov2_vitl14" (berat)
    "pooling":     ["cls", "mean"],
    "temperature": [0.1, 0.2, 0.3],
    "tta_flip":    [False, True],
    "img_size":    [224],                                # + 294 / 336 (kelipatan 14)
    "roi_s_thr":   [0.30],
    "roi_v_thr":   [0.55],
    "roi_pad":     [0.10],
}
keys = list(GRID)
configs = [dict(zip(keys, vals)) for vals in itertools.product(*[GRID[k] for k in keys])]
print(f"{len(paths)} foto uji | {len(anchor_paths)} anchor | {len(configs)} kombinasi config")

## 2. Fungsi (ROI, model cache, embedding cache)

In [ ]:
IMAGENET_MEAN = np.array([0.485,0.456,0.406], np.float32)
IMAGENET_STD  = np.array([0.229,0.224,0.225], np.float32)

def teeth_roi_box(rgb01, s_thr, v_thr, pad, dens=0.12):
    mx = rgb01.max(-1); mn = rgb01.min(-1); v = mx
    s = np.where(mx > 1e-6, (mx-mn)/(mx+1e-6), 0.0)
    teeth = (s < s_thr) & (v > v_thr); H, W = teeth.shape
    col, row = teeth.mean(0), teeth.mean(1)
    def center(fw=0.8, fh=0.6):
        cw,ch=int(W*fw),int(H*fh); x0=(W-cw)//2; y0=(H-ch)//2; return (x0,y0,x0+cw,y0+ch)
    if col.max()<1e-3 or row.max()<1e-3: return center()
    cx=np.where(col>dens*col.max())[0]; ry=np.where(row>dens*row.max())[0]
    x0,x1,y0,y1=int(cx.min()),int(cx.max()),int(ry.min()),int(ry.max())
    bw,bh=x1-x0,y1-y0
    if bw<0.25*W or bh<0.15*H: return center()
    pw,ph=int(bw*pad),int(bh*pad)
    return (max(0,x0-pw),max(0,y0-ph),min(W,x1+pw),min(H,y1+ph))

def load_roi(path, img_size, s_thr, v_thr, pad):
    im = Image.open(path).convert("RGB")
    box = teeth_roi_box(np.asarray(im, np.float32)/255., s_thr, v_thr, pad)
    return np.asarray(im.crop(box).resize((img_size, img_size)), np.float32)/255.

def to_tensor(arr):
    x = (arr - IMAGENET_MEAN)/IMAGENET_STD
    return torch.from_numpy(x).permute(2,0,1).float()

_models = {}
def get_model(enc):
    if enc not in _models:
        print("  loading", enc, "...")
        _models[enc] = torch.hub.load("facebookresearch/dinov2", enc).eval().to(DEVICE)
    return _models[enc]

_emb = {}
@torch.no_grad()
def embed_all(enc, pooling, img_size, s_thr, v_thr, pad, tta):
    key = (enc, pooling, img_size, s_thr, v_thr, pad, tta)
    if key in _emb: return _emb[key]
    m = get_model(enc)
    def feat(arr):
        x = to_tensor(arr).unsqueeze(0).to(DEVICE)
        out = m.forward_features(x)
        return out["x_norm_patchtokens"][0].mean(0) if pooling=="mean" else out["x_norm_clstoken"][0]
    def emb(path):
        arr = load_roi(path, img_size, s_thr, v_thr, pad)
        f = feat(arr)
        if tta: f = f + feat(np.ascontiguousarray(arr[:, ::-1, :]))
        v = f.float().cpu().numpy(); return v/(np.linalg.norm(v)+1e-8)
    res = {p: emb(p) for p in (paths + anchor_paths)}
    _emb[key] = res
    return res

def cfg_id(c):
    return f"{c['encoder'].replace('dinov2_','')}|{c['pooling']}|t{c['temperature']}|{'TTA' if c['tta_flip'] else 'noTTA'}"

## 3. Jalankan sweep

In [ ]:
results = {}   # cfg_id -> {photo_name: AC_weighted}
for i, c in enumerate(configs):
    E = embed_all(c["encoder"], c["pooling"], c["img_size"], c["roi_s_thr"], c["roi_v_thr"], c["roi_pad"], c["tta_flip"])
    anc = np.stack([E[p] for p in anchor_paths])
    col = {}
    for p, nm in zip(paths, names):
        sims = cosine_similarity(E[p][None], anc)[0]
        w = np.exp(sims / c["temperature"]); w /= w.sum()
        col[nm] = float(np.dot(w, anchor_grades))
    results[cfg_id(c)] = col
    print(f"[{i+1}/{len(configs)}] {cfg_id(c)}  done")

df = pd.DataFrame(results)          # baris = foto, kolom = config
df.index.name = "photo"
df = df.round(2)
df.head()

## 4. Weak-label ranking (kompas: config mana paling selaras keparahan)

In [ ]:
SEVERITY = [("normal",1),("diastema",3),("open bite",5),("openbite",5),
            ("crowding",5),("midline",5),("crossbite",6),("deep bite",6),
            ("protrusion",6),("kelas 3",8)]
def weak_severity(name):
    n=name.lower(); sc=1
    for kw,w in SEVERITY:
        if kw in n: sc=max(sc,w)
    return sc
weak = np.array([weak_severity(n) for n in names])

rank_rows = []
for cid in df.columns:
    rho, _ = spearmanr(weak, df[cid].values)
    rank_rows.append({"config": cid, "weak_spearman": round(rho,3), "AC_mean": round(df[cid].mean(),2)})
ranking = pd.DataFrame(rank_rows).sort_values("weak_spearman", ascending=False).reset_index(drop=True)
print("Config terbaik (Spearman tertinggi vs keparahan-nama):")
ranking.head(10)

## 5. Simpan semua hasil

In [ ]:
OUT = "sweep_results"; os.makedirs(OUT, exist_ok=True)

# a) tabel prediksi + ranking (CSV)
df.to_csv(os.path.join(OUT, "ac_predictions.csv"))
ranking.to_csv(os.path.join(OUT, "ranking.csv"), index=False)

# b) heatmap ringkasan foto x config
plt.figure(figsize=(max(8, len(df.columns)*0.5), max(6, len(df)*0.4)))
plt.imshow(df.values, aspect="auto", cmap="viridis", vmin=1, vmax=10)
plt.yticks(range(len(df)), [f"{i:02d}" for i in range(len(df))], fontsize=8)
plt.xticks(range(len(df.columns)), df.columns, rotation=90, fontsize=7)
plt.colorbar(label="Predicted AC (1-10)"); plt.title("AC prediksi: foto (baris) x config (kolom)")
plt.tight_layout(); plt.savefig(os.path.join(OUT,"ac_heatmap.png"), dpi=110); plt.show()
print("indeks -> nama:"); [print(f"  {i:02d} = {n}") for i,n in enumerate(names)]

# c) ranking sebagai gambar tabel
fig, ax = plt.subplots(figsize=(7, 0.4*len(ranking)+1)); ax.axis("off")
tbl = ax.table(cellText=ranking.values, colLabels=ranking.columns, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
ax.set_title("Ranking config (by weak-label Spearman)", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(OUT,"ranking.png"), dpi=130, bbox_inches="tight"); plt.show()
print("Tersimpan di:", OUT)

## 6. Gambar perbandingan per foto (prediksi AC lintas config)

In [ ]:
def disp(path):   # crop ROI utk ditampilkan (pakai ROI default)
    return load_roi(path, 224, 0.30, 0.55, 0.10)

cids = list(df.columns)
for pi, (p, nm) in enumerate(zip(paths, names)):
    vals = [df.loc[nm, c] for c in cids]
    fig, ax = plt.subplots(1, 2, figsize=(13, max(4, len(cids)*0.32)))
    ax[0].imshow(disp(p)); ax[0].set_title(f"{pi:02d}  {nm[:38]}", fontsize=9); ax[0].axis("off")
    y = range(len(cids))
    bars = ax[1].barh(list(y), vals)
    ax[1].set_yticks(list(y)); ax[1].set_yticklabels(cids, fontsize=7); ax[1].invert_yaxis()
    ax[1].set_xlim(1, 10); ax[1].set_xlabel("Predicted AC"); ax[1].grid(axis="x", alpha=0.3)
    for b, v in zip(bars, vals): ax[1].text(v+0.05, b.get_y()+b.get_height()/2, f"{v:.1f}", va="center", fontsize=7)
    ax[1].set_title("Prediksi AC per config", fontsize=9)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"compare_{pi:02d}.png"), dpi=90); plt.close()
print(f"{len(paths)} gambar perbandingan tersimpan di {OUT}/compare_XX.png")